In [35]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import joblib
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer   # Sac de mots
from sklearn.feature_extraction.text import TfidfVectorizer    # TF-IDF
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS  #Stop words English
)

import nltk
from nltk.stem import WordNetLemmatizer   # Lemmatisation
from nltk.stem.snowball import SnowballStemmer

In [3]:
df=pd.read_csv('scitweets_export.tsv',
sep='\t')
print (df.head())
print (df.shape)
print (df.columns)

   Unnamed: 0            tweet_id  \
0           0  316669998137483264   
1           1  319090866545385472   
2           2  322030931022065664   
3           3  322694830620807168   
4           4  328524426658328576   

                                                text  science_related  \
0  Knees are a bit sore. i guess that's a sign th...                0   
1          McDonald's breakfast stop then the gym 🏀💪                0   
2  Can any Gynecologist with Cancer Experience ex...                1   
3  Couch-lock highs lead to sleeping in the couch...                1   
4  Does daily routine help prevent problems with ...                1   

   scientific_claim  scientific_reference  scientific_context  
0               0.0                   0.0                 0.0  
1               0.0                   0.0                 0.0  
2               1.0                   0.0                 0.0  
3               1.0                   0.0                 0.0  
4               1.

In [23]:
emoticons_str = r"""
    (?:
        [:=;] # Eyes
        [oO\-]? # Nose (optional)
        [D\)\]\(\]/\\OpP] # Mouth
    )"""

#Prise en compte des éléments qui doivent être regroupés
regex_str = [
    emoticons_str,
    r'<[^>]+>', # HTML tags
    r'(?:@[\w_]+)', # @-mentions
    r"(?:\#+[\w_]+[\w\'_\-]*[\w_]+)", # hash-tags
    r'http[s]?://(?:[a-z]|[0-9]|[$-_@.&amp;+]|[!*\(\),]|(?:%[0-9a-f][0-9a-f]))+', # URLs

    r'(?:(?:\d+,?)+(?:\.?\d+)?)', # nombres
    r"(?:[a-z][a-z'\-_]+[a-z])", # mots avec - et '
    r'(?:[\w_]+)', # autres mots
    r'(?:\S)' # le reste
]

tokens_re = re.compile(r'('+'|'.join(regex_str)+')', re.VERBOSE | re.IGNORECASE)
emoticon_re = re.compile(r'^'+emoticons_str+'$', re.VERBOSE | re.IGNORECASE)

def tokenize(s):
    return tokens_re.findall(s)

def preprocess(s, emoticon=True, lowercase=False):
    tokens = tokenize(s)
    if lowercase:
        tokens = [tok.lower() for tok in tokens]
    if not emoticon:
        tokens = [tok for tok in tokens if not emoticon_re.search(tok)]
    return tokens

In [33]:

print(tokenize(df.text[618]))
print(preprocess(df.text[618], False, True))

['#VIP2', 'Box', 'office', 'Hit', 'With', 'Youths', ',', 'Kids', '&', 'Family', 'Audience', 'Full', 'Support', '#VIP2BlockBuster3RdWeek', '#VIP2BB3rdWeek', 'https://t.co/6VWYClcAtt']
['#vip2', 'box', 'office', 'hit', 'with', 'youths', ',', 'kids', '&', 'family', 'audience', 'full', 'support', '#vip2blockbuster3rdweek', '#vip2bb3rdweek', 'https://t.co/6vwyclcatt']


In [32]:
contractions_map = {
    "don't": "do not",
    "doesn't": "does not",
    "can't": "cannot",
    "won't": "will not",
    "i'm": "i am",
    "it's": "it is",
    "you're": "you are",
    "they're": "they are",
    "we're": "we are",
}

def expand_contractions(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []
    for t in tokens:
        key = t.lower()
        if key in contractions_map:
            expanded.extend(contractions_map[key].split())
        else:
            expanded.append(t)
    return " ".join(expanded)

def stop_words(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []
    for t in tokens:
        key = t.lower()
        if key not in ENGLISH_STOP_WORDS:
            expanded.append(t)
    return " ".join(expanded)



text = "I’m happy but I don't know why."
print(expand_contractions(text))

print(stop_words(text))



i am happy but I do not know why.
want cake


In [56]:
#nltk.download('wordnet')
stemmer = nltk.stem.porter.PorterStemmer()#PorterStemmer()
lemmatizer = WordNetLemmatizer()


def stemmerLemma(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []
    for t in tokens:
        key = t.lower()
        stem = stemmer.stem(key)
        lemma = lemmatizer.lemmatize(stem)
        expanded.append(lemma)
    return " ".join(expanded)

print(df.text[3])
print(stemmerLemma(df.text[3]))
text = "I want this cake"
print(stop_words(stemmerLemma(text)))
print(stemmerLemma("I am running"))

Couch-lock highs lead to sleeping in the couch. Gotta stop doing this shit.
couch-lock high lead to sleep in the couch. gotta stop do thi shit.
want thi cake
i am run
